<a href="https://colab.research.google.com/github/Diggi14/project_Property2/blob/main/model_selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/Diggi14/project_Property2.git

Cloning into 'project_Property2'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 49 (delta 22), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 3.44 MiB | 6.44 MiB/s, done.
Resolving deltas: 100% (22/22), done.


In [3]:
!pip install xgboost

In [32]:
import pandas as pd
from sklearn.model_selection import train_test_split,KFold,cross_val_score
from sklearn.preprocessing import StandardScaler,OrdinalEncoder,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor,GradientBoostingRegressor,AdaBoostRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
from sklearn.decomposition import PCA

In [5]:
df=pd.read_csv('/content/project_Property2/model_trainer.csv')

In [6]:
df.head(5)

,Unnamed: 0,BEDROOM_NUM,BATHROOM_NUM,BALCONY_NUM,PRICE_PER_UNIT_AREA,FURNISH,AGE,FLOOR_NUM,SUPERBUILTUP_SQFT,PRICE,LOCALITY_WO_CITY
0,0,4.0,4.0,4.0,8766.0,Semi-furnished,5-10,high,3434.0,2.63,Sector 84
1,1,4.0,4.0,3.0,21176.0,Semi-furnished,1-5,mid,2870.0,3.60,Sector 81
2,2,3.0,3.0,3.0,13740.0,Semi-furnished,1-5,high,2802.0,3.85,Sector 112
3,3,3.0,4.0,4.0,8515.0,Semi-furnished,1-5,mid,2290.0,1.95,Sector 104
4,4,2.0,2.0,3.0,11571.0,Unfurnished,0-1,high,1400.0,1.62,Sector 74


In [7]:
df.drop('Unnamed: 0',axis=1,inplace=True)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9330 entries, 0 to 9329
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   BEDROOM_NUM          9330 non-null   float64
 1   BATHROOM_NUM         9330 non-null   float64
 2   BALCONY_NUM          9330 non-null   float64
 3   PRICE_PER_UNIT_AREA  9330 non-null   float64
 4   FURNISH              9330 non-null   object 
 5   AGE                  9330 non-null   object 
 6   FLOOR_NUM            9330 non-null   object 
 7   SUPERBUILTUP_SQFT    9330 non-null   float64
 8   PRICE                9330 non-null   float64
 9   LOCALITY_WO_CITY     9330 non-null   object 
dtypes: float64(6), object(4)
memory usage: 729.0+ KB


In [9]:
X=df.drop('PRICE',axis=1)
y=df['PRICE']

In [13]:
import numpy as np
y=np.log1p(y)

Ordinary encoding

In [20]:
preprocessor=ColumnTransformer(transformers=[
    ('ordinal_encoder',OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),['FURNISH','AGE','FLOOR_NUM','LOCALITY_WO_CITY']),
    ('stdscale',StandardScaler(),['BEDROOM_NUM','BATHROOM_NUM','BALCONY_NUM','PRICE_PER_UNIT_AREA','SUPERBUILTUP_SQFT'])],
    remainder='passthrough'
)

In [21]:
pipeline=Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('model',LinearRegression())
    ]
)

In [22]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X,y, cv=kfold, scoring='r2')

In [23]:
scores.mean(),scores.std()

(np.float64(0.9471032663645538), np.float64(0.004781133785337882))

In [25]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [26]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ordinal_encoder',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['FURNISH', 'AGE',
                                                   'FLOOR_NUM',
                                                   'LOCALITY_WO_CITY']),
                                                 ('stdscale', StandardScaler(),
                                                  ['BEDROOM_NUM',
                                                   'BATHROOM_NUM',
                                                   'BALCONY_NUM',
                                                   'PRICE_PER_UNIT_AREA',
                                                   'SUPERBUILTUP_SQFT'])])),
                ('model', LinearRegression())])

In [27]:
y_pred=np.expm1(pipeline.predict(X_test))

In [29]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.28813216911417905

In [39]:
def score(model_name,model):
  output=[]
  output.append(model_name)

  preprocessor=ColumnTransformer(transformers=[
    ('ordinal_encoder',OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),['FURNISH','AGE','FLOOR_NUM','LOCALITY_WO_CITY']),
    ('stdscale',StandardScaler(),['BEDROOM_NUM','BATHROOM_NUM','BALCONY_NUM','PRICE_PER_UNIT_AREA','SUPERBUILTUP_SQFT'])],
    remainder='passthrough')

  pipeline=Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('model',model)
    ])

  kfold = KFold(n_splits=10, shuffle=True, random_state=42)
  scores = cross_val_score(pipeline, X,y, cv=kfold, scoring='r2')

  output.append(scores.mean())

  X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
  pipeline.fit(X_train,y_train)
  y_pred=np.expm1(pipeline.predict(X_test))
  output.append(mean_absolute_error(np.expm1(y_test),y_pred))

  return output

In [40]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [41]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(score(model_name, model))

In [42]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [43]:
model_df.sort_values(by='mae')

,name,r2,mae
6,extra trees,0.981466,0.116454
5,random forest,0.980208,0.120698
10,xgboost,0.981301,0.135649
4,decision tree,0.965974,0.152395
7,gradient boosting,0.975053,0.159433
9,mlp,0.965077,0.198054
2,ridge,0.947103,0.288108
0,linear_reg,0.947103,0.288132
1,svr,0.923535,0.348231
8,adaboost,0.894836,0.409607


# One hot encoding

In [44]:
def score(model_name,model):
  output=[]
  output.append(model_name)

  preprocessor=ColumnTransformer(transformers=[
    ('ordinal_encoder',OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),['FLOOR_NUM']),
    ('onehot',OneHotEncoder(handle_unknown='ignore'),['LOCALITY_WO_CITY','FURNISH','AGE']),
    ('stdscale',StandardScaler(),['BEDROOM_NUM','BATHROOM_NUM','BALCONY_NUM','PRICE_PER_UNIT_AREA','SUPERBUILTUP_SQFT'])],
    remainder='passthrough')

  pipeline=Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('model',model)
    ])

  kfold = KFold(n_splits=10, shuffle=True, random_state=42)
  scores = cross_val_score(pipeline, X,y, cv=kfold, scoring='r2')

  output.append(scores.mean())

  X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
  pipeline.fit(X_train,y_train)
  y_pred=np.expm1(pipeline.predict(X_test))
  output.append(mean_absolute_error(np.expm1(y_test),y_pred))

  return output

In [45]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [46]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(score(model_name, model))
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
model_df.sort_values(by='mae')

,name,r2,mae
6,extra trees,0.982523,0.102048
5,random forest,0.980083,0.111313
10,xgboost,0.981087,0.135777
4,decision tree,0.969208,0.136770
7,gradient boosting,0.975397,0.160229
9,mlp,0.974665,0.181564
1,svr,0.969700,0.238357
2,ridge,0.956018,0.250203
0,linear_reg,0.956063,0.250531
8,adaboost,0.895205,0.426113


# onehot with pca

In [52]:
def score(model_name,model):
  output=[]
  output.append(model_name)

  preprocessor=ColumnTransformer(transformers=[
    ('ordinal_encoder',OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),['FLOOR_NUM']),
    ('onehot',OneHotEncoder(handle_unknown='ignore', sparse_output=False),['LOCALITY_WO_CITY','FURNISH','AGE']),
    ('stdscale',StandardScaler(),['BEDROOM_NUM','BATHROOM_NUM','BALCONY_NUM','PRICE_PER_UNIT_AREA','SUPERBUILTUP_SQFT'])],
    remainder='passthrough')

  pipeline=Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('pca',PCA(n_components=0.95, svd_solver='full')),
        ('model',model)
    ])

  kfold = KFold(n_splits=10, shuffle=True, random_state=42)
  scores = cross_val_score(pipeline, X,y, cv=kfold, scoring='r2')

  output.append(scores.mean())

  X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
  pipeline.fit(X_train,y_train)
  y_pred=np.expm1(pipeline.predict(X_test))
  output.append(mean_absolute_error(np.expm1(y_test),y_pred))

  return output

In [53]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [54]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(score(model_name, model))
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
model_df.sort_values(by='mae')

,name,r2,mae
6,extra trees,0.970737,0.188798
9,mlp,0.969544,0.200559
10,xgboost,0.968425,0.217186
5,random forest,0.962144,0.221049
7,gradient boosting,0.960863,0.236355
1,svr,0.966383,0.247132
0,linear_reg,0.950219,0.271165
2,ridge,0.950222,0.271200
4,decision tree,0.914942,0.338505
8,adaboost,0.861938,0.493202


In [55]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 2.1 MB/s eta 0:00:00


In [56]:
import category_encoders as ce
def score(model_name,model):
  output=[]
  output.append(model_name)

  preprocessor=ColumnTransformer(transformers=[
    ('ordinal_encoder',OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),['FLOOR_NUM']),
    ('onehot',OneHotEncoder(handle_unknown='ignore', sparse_output=False),['FURNISH','AGE']),
    ('stdscale',StandardScaler(),['BEDROOM_NUM','BATHROOM_NUM','BALCONY_NUM','PRICE_PER_UNIT_AREA','SUPERBUILTUP_SQFT']),
    ('target',ce.TargetEncoder(), ['LOCALITY_WO_CITY'])],
    remainder='passthrough')

  pipeline=Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('pca',PCA(n_components=0.95, svd_solver='full')),
        ('model',model)
    ])

  kfold = KFold(n_splits=10, shuffle=True, random_state=42)
  scores = cross_val_score(pipeline, X,y, cv=kfold, scoring='r2')

  output.append(scores.mean())

  X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
  pipeline.fit(X_train,y_train)
  y_pred=np.expm1(pipeline.predict(X_test))
  output.append(mean_absolute_error(np.expm1(y_test),y_pred))

  return output

In [57]:
model_output2 = []
for model_name,model in model_dict.items():
    model_output2.append(score(model_name, model))
model_df2 = pd.DataFrame(model_output, columns=['name','r2','mae'])
model_df2.sort_values(by='mae')

,name,r2,mae
6,extra trees,0.970737,0.188798
9,mlp,0.969544,0.200559
10,xgboost,0.968425,0.217186
5,random forest,0.962144,0.221049
7,gradient boosting,0.960863,0.236355
1,svr,0.966383,0.247132
0,linear_reg,0.950219,0.271165
2,ridge,0.950222,0.271200
4,decision tree,0.914942,0.338505
8,adaboost,0.861938,0.493202


1.   one hot encoding perform best
2.   one hot encoding with pca perform 2nd best


# **using random forest with one hot encoding**

In [59]:
from sklearn.model_selection import GridSearchCV

In [60]:
param_grid = {
    'regressor__n_estimators': [100, 200, 300, 500],

    'regressor__max_depth': [None, 10, 20, 30, 40],

    'regressor__max_samples': [None, 0.5, 0.7, 0.9],

    'regressor__max_features': ['sqrt', 'log2', None],

    'regressor__bootstrap': [True, False],

    'regressor__criterion': ['squared_error', 'absolute_error', 'poisson'],

    'regressor__min_samples_split': [2, 5, 10],

    'regressor__min_samples_leaf': [1, 2, 4],

    'regressor__ccp_alpha': [0.0, 0.001, 0.01],

    'regressor__warm_start': [False, True]
}

In [71]:
preprocessor=ColumnTransformer(transformers=[
    ('ordinal_encoder',OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),['FURNISH','AGE','FLOOR_NUM','LOCALITY_WO_CITY']),
    ('stdscale',StandardScaler(),['BEDROOM_NUM','BATHROOM_NUM','BALCONY_NUM','PRICE_PER_UNIT_AREA','SUPERBUILTUP_SQFT'])],
    remainder='passthrough')

pipeline=Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('regressor',RandomForestRegressor(random_state=42,
    n_jobs=-1))
    ])

In [73]:
from sklearn.model_selection import RandomizedSearchCV
search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)

In [76]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [77]:
search.fit(X_train,y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
40 fits failed out of a total of 100.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
40 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py", line 662, in fit
    self._final_estimator.fit(Xt, y, **l

RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(remainder='passthrough',
                                                                transformers=[('ordinal_encoder',
                                                                               OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                              unknown_value=-1),
                                                                               ['FURNISH',
                                                                                'AGE',
                                                                                'FLOOR_NUM',
                                                                                'LOCALITY_WO_CITY']),
                                                                              ('stdscale',
                                                                               StandardScaler(),
                                                                               ['BEDROOM_NUM',
                                                                                'BATHROOM_NUM',
                                                                                'BALCONY_NUM',
                                                                                'PRICE_PER_UNIT_AR...
                                                                 'poisson'],
                                        'regressor__max_depth': [None, 10, 20,
                                                                 30, 40],
                                        'regressor__max_features': ['sqrt',
                                                                    'log2',
                                                                    None],
                                        'regressor__max_samples': [None, 0.5,
                                                                   0.7, 0.9],
                                        'regressor__min_samples_leaf': [1, 2,
                                                                        4],
                                        'regressor__min_samples_split': [2, 5,
                                                                         10],
                                        'regressor__n_estimators': [100, 200,
                                                                    300, 500],
                                        'regressor__warm_start': [False, True]},
                   random_state=42, scoring='neg_mean_absolute_error')

In [79]:
best_model = search.best_estimator_

print("Best Parameters:")
print(search.best_params_)

Best Parameters:
{'regressor__warm_start': False, 'regressor__n_estimators': 300, 'regressor__min_samples_split': 10, 'regressor__min_samples_leaf': 4, 'regressor__max_samples': 0.9, 'regressor__max_features': None, 'regressor__max_depth': None, 'regressor__criterion': 'squared_error', 'regressor__ccp_alpha': 0.0, 'regressor__bootstrap': True}


In [80]:
y_pred = best_model.predict(X_test)

from sklearn.metrics import r2_score, mean_absolute_error

print("R2 Score:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))

R2 Score: 0.9780392350146335
MAE: 0.03238405843256016


# **exporting model**

In [81]:
preprocessor=ColumnTransformer(transformers=[
    ('ordinal_encoder',OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),['FURNISH','AGE','FLOOR_NUM','LOCALITY_WO_CITY']),
    ('stdscale',StandardScaler(),['BEDROOM_NUM','BATHROOM_NUM','BALCONY_NUM','PRICE_PER_UNIT_AREA','SUPERBUILTUP_SQFT'])],
    remainder='passthrough')

pipeline=Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('regressor',RandomForestRegressor(random_state=42,n_jobs=-1,n_estimators=300,warm_start= False, min_samples_split= 10,min_samples_leaf= 4,max_samples=0.9,criterion='squared_error',ccp_alpha= 0.0,bootstrap= True))
    ])

In [82]:
pipeline.fit(X,y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ordinal_encoder',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['FURNISH', 'AGE',
                                                   'FLOOR_NUM',
                                                   'LOCALITY_WO_CITY']),
                                                 ('stdscale', StandardScaler(),
                                                  ['BEDROOM_NUM',
                                                   'BATHROOM_NUM',
                                                   'BALCONY_NUM',
                                                   'PRICE_PER_UNIT_AREA',
                                                   'SUPERBUILTUP_SQFT'])])),
                ('regressor',
                 RandomForestRegressor(max_samples=0.9, min_samples_leaf=4,
                                       min_samples_split=10, n_estimators=300,
                                       n_jobs=-1, random_state=42))])

In [84]:
import pickle
with open('pipeline.pkl','wb') as f:
  pickle.dump(pipeline,f)

In [85]:
with open('df.pkl','wb') as f:
  pickle.dump(X,f)